   
## Real-Time Fraud Detection Pipeline — Hands-On Lab
### Auto Loader → Bronze → Silver → AI Scoring

### Scenario
Your fintech platform processes **millions of transaction events** daily — payments, transfers, refunds, and fraud checks. These events originate from an upstream system (Amazon Kinesis) and land as JSON files in cloud storage.

Your job: build a **full medallion pipeline** — from raw file ingestion through real-time enrichment to AI-powered fraud scoring — using Auto Loader, Structured Streaming, and Model Serving on Databricks.

### 🧪 How This Lab Works
This notebook contains **5 coding exercises** where key parts of the code have been removed. Look for `<FILL_IN>` placeholders — replace them with the correct code to make each cell run successfully.

| Exercise | Cell Title | Concept |
|---|---|---|
| 1 | Auto Loader: Ingest JSON into Bronze Delta table | Auto Loader options and streaming triggers |
| 2 | Silver enrichment stream (Bronze to Silver) | Feature engineering with PySpark |
| 3 | Velocity aggregation stream (10-min windows) | Watermarks and windowed aggregations |
| 4 | Score Silver transactions with ai_query() | Calling a model serving endpoint from SQL |
| 5 | UDF-based fraud scoring (no endpoint) | Loading an MLflow model as a Spark UDF |

> **💡 Tip:** Read the markdown explanations above each exercise — they contain the concepts you need. Full answers are in the **Lab Instructions** notebook.

### Environment Setup
The Volume at `/Volumes/fintech_demo/bronze_layer/kinesis_landing` acts as our simulated Kinesis Firehose delivery path.

> **Note:** In production, this Volume path would be an external Volume pointing to the S3 prefix where Kinesis Data Firehose writes JSON.

In [0]:
# ---------------------------------------------------------------------------
# Shared configuration — MUST match the producer notebook
# ---------------------------------------------------------------------------
CATALOG   = "fintech_demo"
SCHEMA    = "bronze_layer"
VOLUME    = "kinesis_landing"          # simulated Kinesis Firehose delivery path
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.txn_events_bronze"

# Checkpoint stored inside the Volume (fine for demo; use a separate path in prod)
CHECKPOINT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoint"

# Landing zone where JSON files arrive
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/events"

print(f"\u2705 Catalog  : {CATALOG}")
print(f"\u2705 Schema   : {CATALOG}.{SCHEMA}")
print(f"\u2705 Volume   : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
print(f"\u2705 Landing  : {LANDING_PATH}")
print(f"\u2705 Bronze   : {BRONZE_TABLE}")

In [0]:
# Ensure the landing zone exists (producer notebook creates the Volume)
import os
os.makedirs(LANDING_PATH, exist_ok=True)

print(f"\u2705 Producer configured")
print(f"   Landing zone: {LANDING_PATH}")

### Peek at the Landing Zone
Before we ingest, let’s see what the Producer notebook has dropped into our landing zone. If the producer is running, you should see JSON files accumulating. Each file contains 50 newline-delimited JSON events — exactly what Kinesis Data Firehose would deliver.

> **Re-run this cell** at any time to see new files appear.

In [0]:
# ---------------------------------------------------------------------------
# Check what files the Producer notebook has delivered so far
# ---------------------------------------------------------------------------
import os

try:
    files = sorted([f for f in os.listdir(LANDING_PATH) if f.endswith(".json")])
    print(f"\U0001F4C2 Landing zone: {LANDING_PATH}")
    print(f"   Total JSON files: {len(files)}\n")
    for f in files[-10:]:  # show last 10 files
        size = os.path.getsize(os.path.join(LANDING_PATH, f))
        print(f"   \u2022 {f}  ({size:,} bytes)")
    if len(files) > 10:
        print(f"   ... and {len(files) - 10} earlier files")
    if len(files) == 0:
        print("   \u26a0\ufe0f  No files yet! Make sure the Producer notebook is running in another tab.")
except FileNotFoundError:
    print("\u26a0\ufe0f  Landing zone not found. Run Step 0 first, then start the Producer notebook.")

### Ingest into Bronze with Auto Loader

This is the core of the demo. We use `cloudFiles` (Auto Loader) to:

1. **Discover** all JSON files in the landing zone
2. **Infer the schema** automatically (no manual DDL needed)
3. **Add ingestion metadata** — `_metadata.file_path` and `_metadata.file_modification_time` come free with Auto Loader
4. **Write to a Delta table** with exactly-once guarantees via checkpointing

Key options:
* `cloudFiles.format` — `json` (matches Kinesis Firehose NDJSON output)
* `cloudFiles.schemaLocation` — persists the inferred schema so it's stable across runs
* `cloudFiles.inferColumnTypes` — infer actual types instead of defaulting everything to STRING
* `trigger(availableNow=True)` — processes all pending files then stops (required on serverless)

> **Re-run the cell below** each time you want to catch up with the producer. Each run picks up only NEW files.

In [0]:
# ---------------------------------------------------------------------------
# EXERCISE 1: Auto Loader — Bronze ingestion from the Kinesis landing zone
# ---------------------------------------------------------------------------
# Auto Loader uses the "cloudFiles" format to incrementally discover and
# ingest new files. Complete the missing .option() values and trigger config.
#
# Replace each <FILL_IN> with the correct value to make this cell run.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, lit

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", <FILL_IN>)                    # What file format are we reading from the landing zone?
    .option("cloudFiles.schemaLocation", <FILL_IN>)            # Where should Auto Loader persist the inferred schema?
    .option("cloudFiles.inferColumnTypes", <FILL_IN>)          # Should Auto Loader infer real types instead of all strings?
    .option("cloudFiles.schemaEvolutionMode", <FILL_IN>)       # How should new upstream fields be handled?
    .load(LANDING_PATH)
)

# Add Bronze audit columns (provided for you)
bronze_stream = (
    raw_stream
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source", lit("kinesis_firehose_sim"))
)

# Write to Bronze Delta table
query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(<FILL_IN>)                                        # Process all available files then stop (serverless-compatible)
    .toTable(BRONZE_TABLE)
)

# Wait for the micro-batch to finish
query.awaitTermination()
print(f"\u2705 Auto Loader stream completed")
print(f"   Bronze table: {BRONZE_TABLE}")

### Verify the Bronze Table
Let’s inspect what Auto Loader wrote. Notice the automatically inferred schema, the audit columns, and the event count matching what the producer delivered.

In [0]:
# ---------------------------------------------------------------------------
# Quick Bronze table inspection
# ---------------------------------------------------------------------------
bronze_df = spark.table(BRONZE_TABLE)

print(f"\U0001F4CA Row count: {bronze_df.count():,}")
print(f"\n\U0001F4DD Schema:")
bronze_df.printSchema()

In [0]:
# ---------------------------------------------------------------------------
# Preview a sample of Bronze records
# ---------------------------------------------------------------------------
display(
    spark.sql(f"""
        SELECT event_id, event_type, timestamp, amount, currency,
               sender_id, receiver_id, merchant, status, channel,
               _ingest_timestamp, _source
        FROM {BRONZE_TABLE}
        ORDER BY timestamp DESC
        LIMIT 1000
    """)
)

### Silver Layer: Real-Time Enrichment and Fraud Signals

In the medallion architecture, the **Silver layer** transforms raw Bronze data into curated, business-ready datasets. For fintech fraud detection, this means:

| Silver Table | Purpose | Technique |
|---|---|---|
| `txn_events_silver` | Cleaned, deduplicated, enriched transactions | Structured Streaming from Bronze |
| `txn_velocity_1min` | Transaction velocity per sender in 1-min windows | Streaming windowed aggregation with watermarks |

**Why streaming?** These Silver tables update every time you run the Auto Loader consumer. In production with a continuous job, they would stay within minutes of the live data — enabling near-real-time fraud alerting.

In [0]:
# ---------------------------------------------------------------------------
# Silver layer configuration
# ---------------------------------------------------------------------------
SILVER_SCHEMA     = "silver_layer"
SILVER_TABLE      = f"{CATALOG}.{SILVER_SCHEMA}.txn_events_silver"
VELOCITY_TABLE    = f"{CATALOG}.{SILVER_SCHEMA}.txn_velocity_1min"

# Separate checkpoints for each stream (mandatory — never share checkpoints)
SILVER_CHECKPOINT   = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoint_silver"
VELOCITY_CHECKPOINT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoint_velocity"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

print(f"\u2705 Silver schema  : {CATALOG}.{SILVER_SCHEMA}")
print(f"\u2705 Silver table   : {SILVER_TABLE}")
print(f"\u2705 Velocity table : {VELOCITY_TABLE}")

In [0]:
# ---------------------------------------------------------------------------
# EXERCISE 2: Silver Enrichment — stream from Bronze, clean, and add fraud signals
# ---------------------------------------------------------------------------
# The Silver layer turns raw data into curated datasets. Your task is to
# derive four fraud-detection feature columns from the Bronze data.
#
# Replace each <FILL_IN> with the correct PySpark expression.
# The functions you need are already imported below.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import (
    col, to_timestamp, hour, dayofweek, when, coalesce, lit, current_timestamp
)

bronze_stream = spark.readStream.table(BRONZE_TABLE)

silver_stream = (
    bronze_stream
    # Parse the ISO-8601 string into a proper TimestampType
    .withColumn("event_ts", to_timestamp("timestamp"))
    # Derived fraud-relevant features
    .withColumn("hour_of_day", <FILL_IN>)                      # Extract the hour (0-23) from event_ts
    .withColumn("day_of_week", <FILL_IN>)                      # Extract the day of the week from event_ts (1=Sun … 7=Sat)
    .withColumn("is_high_value", <FILL_IN>)                    # True when amount exceeds 5000, otherwise False
    .withColumn("is_off_hours", when(
        <FILL_IN>, True                                        # True when the hour is before 6 AM or after 10 PM
    ).otherwise(False))
    # Clean nulls
    .withColumn("merchant", coalesce(col("merchant"), lit("N/A")))
    # Drop raw string timestamp (replaced by event_ts)
    .drop("timestamp")
    # Deduplicate on event_id (exactly-once at Silver level)
    .dropDuplicates(["event_id"])
)

query = (
    silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", SILVER_CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE)
)

query.awaitTermination()
print(f"\u2705 Silver enrichment stream completed")
print(f"   Silver table: {SILVER_TABLE}")
print(f"   Row count:    {spark.table(SILVER_TABLE).count():,}")

In [0]:
# ---------------------------------------------------------------------------
# Inspect the Silver enriched table
# ---------------------------------------------------------------------------
print("\U0001F4DD Silver schema (note the derived fraud-signal columns):")
spark.table(SILVER_TABLE).printSchema()

display(
    spark.sql(f"""
        SELECT event_id, event_type, event_ts, amount, currency,
               sender_id, receiver_id, merchant, status, channel,
               hour_of_day, day_of_week, is_high_value, is_off_hours
        FROM {SILVER_TABLE}
        ORDER BY event_ts DESC
        LIMIT 10
    """)
)

#### Velocity Checks — Streaming Windowed Aggregation

We use a **1-minute tumbling window** grouped by `sender_id` to compute transaction velocity. This is a stateful Structured Streaming operation:

* **Watermark** (`1 minute`) — tells Spark how late data can arrive before a window is finalized
* **Tumbling window** (`1 minute`) — non-overlapping time buckets
* **Append output mode** — results are emitted only after the watermark passes the window’s end, guaranteeing the window is complete

Flags generated:
* `is_velocity_alert` — sender made **3+ transactions** in a single 1-min window
* `is_high_volume` — sender moved **> $15,000** in a single 1-min window

In [0]:
# ---------------------------------------------------------------------------
# EXERCISE 3: Velocity Checks — 1-minute windowed aggregation per sender
# ---------------------------------------------------------------------------
# Velocity checks catch rapid-fire transactions from a single sender.
# Your task is to set the watermark, define the tumbling window, and
# choose the correct output mode.
#
# Replace each <FILL_IN> with the correct expression or value.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import (
    col, to_timestamp, window, count, sum as _sum, max as _max,
    avg as _avg, when, current_timestamp
)

# Read Bronze as a stream (separate stream from Silver enrichment)
velocity_source = (
    spark.readStream
    .table(BRONZE_TABLE)
    .withColumn("event_ts", to_timestamp("timestamp"))
    # Watermark: allow up to 1 minute of late data
    .withWatermark(<FILL_IN>)                                   # Two args: the event-time column name and the lateness threshold
)

# 1-minute tumbling window aggregation per sender
velocity_agg = (
    velocity_source
    .groupBy(
        window(<FILL_IN>),                                      # Two args: the event-time column and the window duration
        "sender_id"
    )
    .agg(
        count("*").alias("txn_count"),
        _sum("amount").alias("total_amount"),
        _max("amount").alias("max_txn_amount"),
        _avg("amount").alias("avg_txn_amount"),
    )
    # Fraud signal flags
    .withColumn("is_velocity_alert",
        when(col("txn_count") >= 2, True).otherwise(False))
    .withColumn("is_high_volume",
        when(col("total_amount") > 15000, True).otherwise(False))
    # Flatten the window struct for easier querying
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "sender_id", "txn_count", "total_amount",
        "max_txn_amount", "avg_txn_amount",
        "is_velocity_alert", "is_high_volume"
    )
)

query = (
    velocity_agg.writeStream
    .format("delta")
    .outputMode(<FILL_IN>)                                      # Which mode emits results only after the watermark passes?
    .option("checkpointLocation", VELOCITY_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(VELOCITY_TABLE)
)

query.awaitTermination()
print(f"\u2705 Velocity aggregation stream completed")
print(f"   Velocity table: {VELOCITY_TABLE}")
print(f"   Window count:   {spark.table(VELOCITY_TABLE).count():,}")

In [0]:
# ---------------------------------------------------------------------------
# Velocity check results — show highest-velocity senders
# ---------------------------------------------------------------------------
print("\U0001F6A8 Top senders by transaction velocity (1-min windows):\n")

display(
    spark.sql(f"""
        SELECT window_start, window_end, sender_id,
               txn_count, ROUND(total_amount, 2) AS total_amount,
               ROUND(max_txn_amount, 2) AS max_txn,
               is_velocity_alert, is_high_volume
        FROM {VELOCITY_TABLE}
        ORDER BY txn_count DESC, total_amount DESC
        LIMIT 5000
    """)
)

### AI-Powered Fraud Scoring

#### With `ai_query()`

The `ai_query()` SQL function calls our **model serving endpoint** to score every Silver transaction in real time. Each row gets a `fraud_score` (0.0–1.0) representing the probability of fraud.

This is the same pattern you'd use in production to:
* Score transactions as they land in Silver
* Feed scores into a Gold alerting table
* Power real-time dashboards and notification systems

In [0]:
# ---------------------------------------------------------------------------
# EXERCISE 4: Real-time fraud scoring via ai_query()
# ---------------------------------------------------------------------------
# ai_query() calls a model serving endpoint from SQL. Your task is to
# complete the function call with the endpoint name and return type.
#
# Replace each <FILL_IN> with the correct SQL value.
# ---------------------------------------------------------------------------
ENDPOINT_NAME = "fintech-fraud-scoring"

scored_df = spark.sql(f"""
    SELECT event_id,
           event_type,
           event_ts,
           amount,
           sender_id,
           receiver_id,
           channel,
           status,
           hour_of_day,
           day_of_week,
           is_high_value,
           is_off_hours,
           ai_query(
               <FILL_IN>,                          -- First argument: the serving endpoint name (use the Python variable)
               named_struct(
                   'amount',        DOUBLE(amount),
                   'hour_of_day',   DOUBLE(hour_of_day),
                   'day_of_week',   DOUBLE(day_of_week),
                   'is_high_value', DOUBLE(CAST(is_high_value AS INT)),
                   'is_off_hours',  DOUBLE(CAST(is_off_hours AS INT))
               ),
               returnType => <FILL_IN>             -- What SQL data type represents a probability (0.0 to 1.0)?
           ) AS fraud_score
    FROM {SILVER_TABLE}
    ORDER BY event_ts DESC
    LIMIT 200
""")

print(f"\U0001F916 Scored {scored_df.count()} transactions via '{ENDPOINT_NAME}'\n")
display(scored_df)

#### With UDF-Based Scoring (No Endpoint Required)

The cell below loads the **same fraud model** from Unity Catalog but applies it as a **Spark UDF** instead of calling the serving endpoint. The model runs directly inside the Spark executors — no network hop, no endpoint cost.

| | `ai_query()` + Endpoint | Spark UDF |
|---|---|---|
| **Latency** | HTTP round-trip per batch | In-process, microseconds/row |
| **Throughput** | Throttled by endpoint concurrency | Scales with cluster parallelism |
| **Cost** | Serving endpoint cost | Free — uses existing compute |
| **Multi-consumer** | Any client (APIs, dashboards, SQL) | Only this Spark job |
| **Best for** | Shared service, large models, A/B testing | Batch scoring, small models, cost-sensitive pipelines |

In [0]:
# ---------------------------------------------------------------------------
# EXERCISE 5: UDF-based fraud scoring — same model, no serving endpoint
# ---------------------------------------------------------------------------
# This loads the fraud model from Unity Catalog as a Spark UDF and applies
# it directly inside the Spark executors. Your task is to load the UDF
# and call it with the correct feature columns.
#
# Replace each <FILL_IN> with the correct expression.
# ---------------------------------------------------------------------------
import mlflow
from pyspark.sql.functions import struct, col

MODEL_NAME = f"{CATALOG}.models.fraud_scorer"

# Reference the model by alias — "Champion" decouples code from version numbers
model_uri = f"models:/{MODEL_NAME}@Champion"

# Load from Unity Catalog as a Spark UDF
fraud_udf = <FILL_IN>                                          # Use mlflow.pyfunc to create a Spark UDF (needs spark, model_uri, and result_type)

# Apply the UDF to the Silver table
silver_df = spark.table(SILVER_TABLE)

scored_udf_df = (
    silver_df
    .withColumn(
        "fraud_score_udf",
        fraud_udf(
            # Pass a struct of the 5 feature columns, each cast to double
            <FILL_IN>                                           # struct() with: amount, hour_of_day, day_of_week, is_high_value, is_off_hours
        ),
    )
    .select(
        "event_id", "event_type", "event_ts", "amount",
        "sender_id", "channel", "is_high_value", "is_off_hours",
        "fraud_score_udf",
    )
    .orderBy(col("fraud_score_udf").desc())
    .limit(200)
)

print(f"\U0001F916 Scored {scored_udf_df.count()} transactions via Spark UDF (no endpoint)\n")
display(scored_udf_df)

### Cleanup

> ⚠️ **Remove all demo resources** (Bronze + Silver + Model + Endpoint). Skip if you want to keep exploring.

> **Don’t forget** to also stop the Producer notebook in the other tab!

In [0]:
# ---------------------------------------------------------------------------
# ⚠️ CLEANUP — Uncomment and run to tear down all demo resources
# ---------------------------------------------------------------------------

# # Serving endpoint (not removed by catalog cascade)
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
try:
    w.serving_endpoints.delete(name="fintech-fraud-scoring")
    print("\U0001F5D1\ufe0f  Deleted serving endpoint: fintech-fraud-scoring")
except Exception:
    print("\u2139\ufe0f  Endpoint already deleted or not found")

# Catalog cascade removes all schemas, tables, volumes, and models
spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")
print("\U0001F5D1\ufe0f  Dropped catalog (cascade): fintech_demo")
print("\U0001F9F9 All demo resources removed")